# AnalystLab Africa — Week 3: SQL & Data Querying
## Dataset 1: Sales Data Analysis
**Intern:** LINDSLY MUYONDA | **Cohort:** Batch B | **Date:** June 2025

---
This notebook analyses the Sample Sales Dataset using SQL queries via `pandasql`.
It covers core SQL, advanced concepts and business problem solving.

In [4]:
!pip install pandasql

In [15]:
import pandas as pd
from pandasql import sqldf


df = pd.read_csv("sales_data_sample.csv", encoding="latin1")
run = lambda q: sqldf(q, {"df": df})

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset loaded: 2823 rows, 25 columns


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [16]:

print(df.dtypes)
print("\nNull values per column:")
print(df.isnull().sum())

ORDERNUMBER           int64
QUANTITYORDERED       int64
PRICEEACH           float64
ORDERLINENUMBER       int64
SALES               float64
ORDERDATE            object
STATUS               object
QTR_ID                int64
MONTH_ID              int64
YEAR_ID               int64
PRODUCTLINE          object
MSRP                  int64
PRODUCTCODE          object
CUSTOMERNAME         object
PHONE                object
ADDRESSLINE1         object
ADDRESSLINE2         object
CITY                 object
STATE                object
POSTALCODE           object
COUNTRY              object
TERRITORY            object
CONTACTLASTNAME      object
CONTACTFIRSTNAME     object
DEALSIZE             object
dtype: object

Null values per column:
ORDERNUMBER            0
QUANTITYORDERED        0
PRICEEACH              0
ORDERLINENUMBER        0
SALES                  0
ORDERDATE              0
STATUS                 0
QTR_ID                 0
MONTH_ID               0
YEAR_ID                0
PRODUCTLINE

##  Core SQL Queries
### Revenue by Product Line
`GROUP BY`, `SUM`, `COUNT`, `ORDER BY`

In [17]:

run("""
SELECT PRODUCTLINE,
       COUNT(DISTINCT ORDERNUMBER) AS total_orders,
       SUM(QUANTITYORDERED)        AS units_sold,
       ROUND(SUM(SALES), 2)        AS total_revenue
FROM df
GROUP BY PRODUCTLINE
ORDER BY total_revenue DESC
""")

,PRODUCTLINE,total_orders,units_sold,total_revenue
0,Classic Cars,199,33992,3919615.66
1,Vintage Cars,175,21069,1903150.84
2,Motorcycles,72,11663,1166388.34
3,Trucks and Buses,73,10777,1127789.84
4,Planes,59,10727,975003.57
5,Ships,65,8127,714437.13
6,Trains,45,2712,226243.47


In [8]:
# Top 10 customers by lifetime  value 
run("""
SELECT CUSTOMERNAME, COUNTRY,
       COUNT(DISTINCT ORDERNUMBER)  AS total_orders,
       ROUND(SUM(SALES), 2)         AS lifetime_value
FROM df
GROUP BY CUSTOMERNAME, COUNTRY
ORDER BY lifetime_value DESC
LIMIT 10
""")

,CUSTOMERNAME,COUNTRY,total_orders,lifetime_value
0,Euro Shopping Channel,Spain,26,912294.11
1,Mini Gifts Distributors Ltd.,USA,17,654858.06
2,"Australian Collectors, Co.",Australia,5,200995.41
3,Muscle Machine Inc,USA,4,197736.94
4,La Rochelle Gifts,France,4,180124.90
5,"Dragon Souveniers, Ltd.",Singapore,5,172989.68
6,Land of Toys Inc.,USA,4,164069.44
7,The Sharp Gifts Warehouse,USA,4,160010.27
8,"AV Stores, Co.",UK,3,157807.81
9,"Anna's Decorations, Ltd",Australia,4,153996.13


### Revenue by Country
Identifying top markets using `GROUP BY` and `COUNT DISTINCT`

In [9]:
run("""
SELECT COUNTRY,
       COUNT(DISTINCT CUSTOMERNAME) AS unique_customers,
       ROUND(SUM(SALES), 2)         AS total_revenue
FROM df
GROUP BY COUNTRY
ORDER BY total_revenue DESC
""")

,COUNTRY,unique_customers,total_revenue
0,USA,35,3627982.83
1,Spain,5,1215686.92
2,France,12,1110916.52
3,Australia,5,630623.10
4,UK,5,478880.46
5,Italy,3,374674.31
6,Finland,3,329581.91
7,Norway,3,307463.70
8,Singapore,2,288488.41
9,Denmark,2,245637.15


## 2.3 Annual Revenue Trend
Using `WHERE`, `GROUP BY`, and aggregate functions to track revenue over time.
> 2005 data is incomplete — only covers Jan–May.

In [10]:
# Annual revenue by year
run("""
SELECT YEAR_ID,
       COUNT(DISTINCT ORDERNUMBER)  AS orders,
       ROUND(SUM(SALES), 2)         AS annual_revenue
FROM df
WHERE STATUS = 'Shipped'
GROUP BY YEAR_ID
ORDER BY YEAR_ID
""")

,YEAR_ID,orders,annual_revenue
0,2003,102,3439718.03
1,2004,139,4528047.22
2,2005,45,1323735.83


##  Advanced SQL Concepts
### Quarterly Revenue Pivot (CASE WHEN)
Using conditional aggregation to pivot quarters into column

In [11]:
# Quarterly revenue pivot
run("""
SELECT YEAR_ID,
       ROUND(SUM(CASE WHEN QTR_ID=1 THEN SALES ELSE 0 END), 2) AS Q1,
       ROUND(SUM(CASE WHEN QTR_ID=2 THEN SALES ELSE 0 END), 2) AS Q2,
       ROUND(SUM(CASE WHEN QTR_ID=3 THEN SALES ELSE 0 END), 2) AS Q3,
       ROUND(SUM(CASE WHEN QTR_ID=4 THEN SALES ELSE 0 END), 2) AS Q4,
       ROUND(SUM(SALES), 2)                                     AS annual_total
FROM df
WHERE STATUS = 'Shipped'
GROUP BY YEAR_ID
ORDER BY YEAR_ID
""")

,YEAR_ID,Q1,Q2,Q3,Q4,annual_total
0,2003,445094.69,562365.22,649514.54,1782743.58,3439718.03
1,2004,833730.68,620484.17,1109396.27,1964436.10,4528047.22
2,2005,973903.28,349832.55,0.00,0.00,1323735.83


## Business Problem Solving
### Top 10 Customers by Lifetime Value
Identifying the highest-value customers for retention strategy.

In [18]:
 
run("""
SELECT CUSTOMERNAME, COUNTRY,
       COUNT(DISTINCT ORDERNUMBER)  AS total_orders,
       ROUND(SUM(SALES), 2)         AS lifetime_value
FROM df
GROUP BY CUSTOMERNAME, COUNTRY
ORDER BY lifetime_value DESC
LIMIT 10
""")

,CUSTOMERNAME,COUNTRY,total_orders,lifetime_value
0,Euro Shopping Channel,Spain,26,912294.11
1,Mini Gifts Distributors Ltd.,USA,17,654858.06
2,"Australian Collectors, Co.",Australia,5,200995.41
3,Muscle Machine Inc,USA,4,197736.94
4,La Rochelle Gifts,France,4,180124.90
5,"Dragon Souveniers, Ltd.",Singapore,5,172989.68
6,Land of Toys Inc.,USA,4,164069.44
7,The Sharp Gifts Warehouse,USA,4,160010.27
8,"AV Stores, Co.",UK,3,157807.81
9,"Anna's Decorations, Ltd",Australia,4,153996.13


## Deal Size Distribution by Country
Understanding whether countries tend toward small, medium, or large deals.

In [12]:

run("""
SELECT COUNTRY,
       SUM(CASE WHEN DEALSIZE='Small'  THEN 1 ELSE 0 END) AS small_deals,
       SUM(CASE WHEN DEALSIZE='Medium' THEN 1 ELSE 0 END) AS medium_deals,
       SUM(CASE WHEN DEALSIZE='Large'  THEN 1 ELSE 0 END) AS large_deals
FROM df
GROUP BY COUNTRY
ORDER BY large_deals DESC
""")

,COUNTRY,small_deals,medium_deals,large_deals
0,USA,435,505,64
1,France,144,149,21
2,Spain,154,171,17
3,Italy,62,44,7
4,Australia,92,86,7
5,Denmark,26,31,6
6,Norway,38,42,5
7,Finland,41,46,5
8,UK,69,71,4
9,Singapore,37,38,4


### Customer Segmentation
Classifying customers into loyalty tiers using a subquery and `CASE WHEN`

In [13]:

run("""
SELECT
    CASE WHEN order_count = 1 THEN 'One-Time'
         WHEN order_count BETWEEN 2 AND 5 THEN 'Occasional'
         ELSE 'Loyal (6+)' END AS segment,
    COUNT(*) AS customer_count,
    ROUND(AVG(total_revenue), 2) AS avg_lifetime_value
FROM (
    SELECT CUSTOMERNAME,
           COUNT(DISTINCT ORDERNUMBER) AS order_count,
           SUM(SALES) AS total_revenue
    FROM df
    GROUP BY CUSTOMERNAME
) sub
GROUP BY 1
ORDER BY avg_lifetime_value DESC
""")

,segment,customer_count,avg_lifetime_value
0,Loyal (6+),2,783576.08
1,Occasional,89,94724.53
2,One-Time,1,34993.92


## 5. Key Insights

KEY INSIGHTS

 1.Top product line is Classic Cars — $3.9M revenue 
 
 2. Top country  is USA — $3.6M, 35 unique customers 
 
  3. Best year is 2004 — $4.5M in shipped orders
 
4. Strongest quarter is  Q4 every year — peak season confirmed 

 5. Most valuable customer is Euro Shopping Channel (Spain) — $912K 
 
6. Loyal customers (6+) has only 2 customers but avg $783K lifetime value |


*Analysis completed using SQL via pandasql in Python

In [3]:
sql_script = """-- 
-- ANALYSTLAB AFRICA - WEEK 3: SQL & DATA QUERYING
-- Dataset: Sample Sales Data
-- Intern: LINDSLY MUYONDA


-- SCHEMA EXPLORATION


SELECT COUNT(*) AS total_rows FROM sales;

SELECT STATUS, COUNT(*) AS count
FROM sales
GROUP BY STATUS
ORDER BY count DESC;


--  CORE SQL QUERIES


-- SELECT, WHERE, ORDER BY
SELECT ORDERNUMBER, CUSTOMERNAME, COUNTRY, PRODUCTLINE, SALES
FROM sales
WHERE STATUS = 'Shipped'
ORDER BY SALES DESC;

SELECT ORDERNUMBER, CUSTOMERNAME, PRODUCTLINE, SALES, DEALSIZE
FROM sales
WHERE DEALSIZE = 'Large'
ORDER BY SALES DESC;

--  GROUP BY, HAVING & Aggregate Functions
SELECT PRODUCTLINE,
       COUNT(DISTINCT ORDERNUMBER)    AS total_orders,
       SUM(QUANTITYORDERED)           AS units_sold,
       ROUND(SUM(SALES), 2)           AS total_revenue,
       ROUND(AVG(SALES), 2)           AS avg_order_value
FROM sales
GROUP BY PRODUCTLINE
ORDER BY total_revenue DESC;

SELECT COUNTRY,
       COUNT(DISTINCT CUSTOMERNAME)   AS unique_customers,
       ROUND(SUM(SALES), 2)           AS total_revenue
FROM sales
GROUP BY COUNTRY
ORDER BY total_revenue DESC;

SELECT YEAR_ID,
       COUNT(DISTINCT ORDERNUMBER)    AS orders,
       ROUND(SUM(SALES), 2)           AS annual_revenue
FROM sales
WHERE STATUS = 'Shipped'
GROUP BY YEAR_ID
ORDER BY YEAR_ID;

SELECT PRODUCTLINE,
       ROUND(SUM(SALES), 2) AS total_revenue
FROM sales
GROUP BY PRODUCTLINE
HAVING SUM(SALES) > 500000
ORDER BY total_revenue DESC;


--  ADVANCED SQL CONCEPTS


-- Quarterly Revenue Pivot (CASE WHEN)
SELECT YEAR_ID,
       ROUND(SUM(CASE WHEN QTR_ID=1 THEN SALES ELSE 0 END), 2) AS Q1,
       ROUND(SUM(CASE WHEN QTR_ID=2 THEN SALES ELSE 0 END), 2) AS Q2,
       ROUND(SUM(CASE WHEN QTR_ID=3 THEN SALES ELSE 0 END), 2) AS Q3,
       ROUND(SUM(CASE WHEN QTR_ID=4 THEN SALES ELSE 0 END), 2) AS Q4,
       ROUND(SUM(SALES), 2)                                     AS annual_total
FROM sales
WHERE STATUS = 'Shipped'
GROUP BY YEAR_ID
ORDER BY YEAR_ID;

-- Subqueries
SELECT CUSTOMERNAME, COUNTRY,
       ROUND(SUM(SALES), 2) AS total_spent
FROM sales
GROUP BY CUSTOMERNAME, COUNTRY
HAVING SUM(SALES) > (
    SELECT AVG(customer_total)
    FROM (
        SELECT SUM(SALES) AS customer_total
        FROM sales
        GROUP BY CUSTOMERNAME
    )
)
ORDER BY total_spent DESC;

SELECT DISTINCT PRODUCTCODE, PRODUCTLINE, MSRP
FROM sales s1
WHERE MSRP > (
    SELECT AVG(MSRP)
    FROM sales s2
    WHERE s2.PRODUCTLINE = s1.PRODUCTLINE
)
ORDER BY PRODUCTLINE, MSRP DESC;

--  Window Functions (RANK, ROW_NUMBER, PARTITION BY)
SELECT CUSTOMERNAME, COUNTRY,
       ROUND(SUM(SALES), 2) AS total_revenue,
       RANK() OVER (
           PARTITION BY COUNTRY
           ORDER BY SUM(SALES) DESC
       ) AS country_rank
FROM sales
GROUP BY CUSTOMERNAME, COUNTRY
ORDER BY COUNTRY, country_rank;

SELECT YEAR_ID, MONTH_ID,
       ROUND(SUM(SALES), 2) AS monthly_revenue,
       ROUND(SUM(SUM(SALES)) OVER (
           PARTITION BY YEAR_ID
           ORDER BY MONTH_ID
       ), 2) AS cumulative_ytd
FROM sales
GROUP BY YEAR_ID, MONTH_ID
ORDER BY YEAR_ID, MONTH_ID;

-- Top 3 products per product line
SELECT PRODUCTLINE, PRODUCTCODE, total_revenue, rn AS rank_in_line
FROM (
    SELECT PRODUCTLINE, PRODUCTCODE,
           ROUND(SUM(SALES), 2) AS total_revenue,
           ROW_NUMBER() OVER (
               PARTITION BY PRODUCTLINE
               ORDER BY SUM(SALES) DESC
           ) AS rn
    FROM sales
    GROUP BY PRODUCTLINE, PRODUCTCODE
)
WHERE rn <= 3
ORDER BY PRODUCTLINE, rn;


-- BUSINESS PROBLEM SOLVING


-- Top 10 customers by lifetime value
SELECT CUSTOMERNAME, COUNTRY,
       COUNT(DISTINCT ORDERNUMBER)    AS total_orders,
       ROUND(SUM(SALES), 2)           AS lifetime_value
FROM sales
GROUP BY CUSTOMERNAME, COUNTRY
ORDER BY lifetime_value DESC
LIMIT 10;

-- Deal size distribution by country
SELECT COUNTRY,
       SUM(CASE WHEN DEALSIZE='Small'  THEN 1 ELSE 0 END) AS small_deals,
       SUM(CASE WHEN DEALSIZE='Medium' THEN 1 ELSE 0 END) AS medium_deals,
       SUM(CASE WHEN DEALSIZE='Large'  THEN 1 ELSE 0 END) AS large_deals
FROM sales
GROUP BY COUNTRY
ORDER BY large_deals DESC;

-- Customer segmentation: loyal vs occasional vs one-time
SELECT
    CASE WHEN order_count = 1            THEN 'One-Time'
         WHEN order_count BETWEEN 2 AND 5 THEN 'Occasional'
         ELSE 'Loyal (6+)' END            AS segment,
    COUNT(*)                              AS customer_count,
    ROUND(AVG(total_revenue), 2)          AS avg_lifetime_value
FROM (
    SELECT CUSTOMERNAME,
           COUNT(DISTINCT ORDERNUMBER) AS order_count,
           SUM(SALES)                  AS total_revenue
    FROM sales
    GROUP BY CUSTOMERNAME
)
GROUP BY 1
ORDER BY avg_lifetime_value DESC;


--  QUERY OPTIMISATION
-- ============================================================

CREATE INDEX IF NOT EXISTS idx_sales_status      ON sales(STATUS);
CREATE INDEX IF NOT EXISTS idx_sales_year        ON sales(YEAR_ID);
CREATE INDEX IF NOT EXISTS idx_sales_productline ON sales(PRODUCTLINE);
CREATE INDEX IF NOT EXISTS idx_sales_customer    ON sales(CUSTOMERNAME);
CREATE INDEX IF NOT EXISTS idx_sales_country     ON sales(COUNTRY);

EXPLAIN QUERY PLAN
SELECT CUSTOMERNAME,
       ROUND(SUM(SALES), 2) AS total_revenue
FROM sales
WHERE STATUS = 'Shipped'
  AND YEAR_ID = 2004
GROUP BY CUSTOMERNAME
ORDER BY total_revenue DESC
LIMIT 10;


"""

with open("sales_analysis.sql", "w") as f:
    f.write(sql_script)

print("sales_analysis.sql has been saved in the same folder as this notebook!")

sales_analysis.sql has been saved in the same folder as this notebook!
